# CTC Model Training Pipeline
This notebook implements the Connectionist Temporal Classification (CTC) pipeline. 
Unlike the sliding window approach, this trains the `CTC_CRNN` sequentially on entire audio recordings using PyTorch's native `CTCLoss`.

In [7]:
import sys

assert sys.version_info >= (3, 10)
IS_COLAB = "google.colab" in sys.modules

if IS_COLAB:
    !git clone https://github.com/stachuapa123/ASR_project.git
    %cd ASR_project
    # !git checkout <YOUR_BRANCH_NAME>  # Uncomment and set this to your branch if needed
    !pip install -q torchmetrics
    from google.colab import drive

    drive.mount("/content/drive")

    # Extract data securely if on Colab
    !mkdir -p "/content/asr_data"
    !unzip -q "/content/drive/MyDrive/asr_data.zip" -d "/content/asr_data"
    DATA_DIR = "/content/asr_data"
else:
    # Local path
    %load_ext autoreload
    %autoreload 2
    DATA_DIR = "../data"  # Update to your local subset or AutorskieDane

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [8]:
import torch
from torch.utils.data import DataLoader

from src.ctc.config import CTCConfig as C
from src.ctc.model import CTCModel
from src.ctc.dataset import CTCDataset, ctc_collate_fn
from src.ctc.augmentation import SpecAugment
from src.ctc.training import train_ctc, EarlyStopping

In [9]:
hparams = {
    "batch_size": 16,
    "epochs": 100,
    "optimizer": "AdamW",
    "lr": 1e-3,
    "weight_decay": 1e-4,
    "max_lr": 3e-3,
    "pct_start": 0.2,
    "early_stopping_patience": 5,
    "early_stopping_min_delta": 0.0,
    "num_workers": 4,
    "specaug_freq_mask": 0.2,
    "specaug_time_mask": 0.125,
    "specaug_p": 0.5,
}

device = C.get_device()
#C.setup_cuda_optimizations()
print(f"Using device: {device}")

Using device: cuda


In [10]:
dataset = CTCDataset(
    data_root=DATA_DIR,
    cache_mode=True,
    apply_augmentations=False,
    max_files=None,
    sample_rate=C.SAMPLE_RATE,
    n_fft=C.N_FFT,
    hop_length=C.HOP_LENGTH,
    n_mels=C.N_MELS,
    standardize=True,
)

n_total = len(dataset)
n_val = max(1, int(0.15 * n_total))
n_train = n_total - n_val
generator = torch.Generator().manual_seed(42)
train_set, val_set = torch.utils.data.random_split(
    dataset,
    [n_train, n_val],
    generator=generator,
)
print(f"Train items: {len(train_set)} | Val items: {len(val_set)}")

train_loader = DataLoader(
    train_set,
    batch_size=hparams["batch_size"],
    shuffle=True,
    collate_fn=ctc_collate_fn,
    num_workers=hparams["num_workers"],
    pin_memory=True,
)
val_loader = DataLoader(
    val_set,
    batch_size=hparams["batch_size"],
    shuffle=False,
    collate_fn=ctc_collate_fn,
    num_workers=hparams["num_workers"],
    pin_memory=True,
)

Pre-computing and caching 10298 files into RAM...


CTC cache: 100%|██████████| 10298/10298 [01:36<00:00, 106.66it/s]

Cache complete. Total items in RAM: 10298
Train items: 8754 | Val items: 1544


In [11]:
model = CTCModel()
criterion = torch.nn.CTCLoss(blank=C.N_CLASSES, zero_infinity=True)
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=hparams["lr"],
    weight_decay=hparams["weight_decay"],
)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=hparams["max_lr"],
    steps_per_epoch=len(train_loader),
    epochs=hparams["epochs"],
    pct_start=hparams["pct_start"],
)
scaler = torch.amp.GradScaler(
    device=device.type,
    enabled=(device.type == "cuda"),
)
es = EarlyStopping(
    patience=hparams["early_stopping_patience"],
    min_delta=hparams["early_stopping_min_delta"],
)
spec_aug = SpecAugment(
    freq_mask_percent=hparams["specaug_freq_mask"],
    time_mask_percent=hparams["specaug_time_mask"],
    p=hparams["specaug_p"],
)

# reuse hparams as checkpoint config (optionally add DATA_DIR, etc.)
config = {
    **hparams,
    "data_root": DATA_DIR,
    "model": "CTCModel",
}

In [12]:
model = train_ctc(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    criterion=criterion,
    device=device,
    n_epochs=hparams["epochs"],
    spec_augment=spec_aug,
    scheduler=scheduler,
    scaler=scaler,
    early_stopping=es,
    save_best_to="../trained_models/CTCModelFirst.pt",
    use_amp=(device.type == "cuda"),
    grad_clip_norm=2.0,
    step_scheduler_per_batch=True,
    checkpoint_config=config,
)

Epoch   1/100 | Train Loss: 3.4032 | Val Loss: 3.0406 | LR: 1.4e-04 [BEST]          
Epoch   2/100 | Train Loss: 2.5915 | Val Loss: 2.1089 | LR: 1.9e-04 [BEST]          
Epoch   3/100 | Train Loss: 1.9539 | Val Loss: 1.6800 | LR: 2.8e-04 [BEST]          
Epoch   4/100 | Train Loss: 1.6562 | Val Loss: 1.4483 | LR: 4.0e-04 [BEST]          
Epoch   5/100 | Train Loss: 1.4882 | Val Loss: 1.3657 | LR: 5.4e-04 [BEST]          
Epoch   6/100 | Train Loss: 1.3618 | Val Loss: 1.2409 | LR: 7.1e-04 [BEST]          
Epoch   7/100 | Train Loss: 1.2589 | Val Loss: 1.1591 | LR: 9.1e-04 [BEST]          
Epoch   8/100 | Train Loss: 1.1771 | Val Loss: 1.0882 | LR: 1.1e-03 [BEST]          
Epoch   9/100 | Train Loss: 1.1003 | Val Loss: 1.0448 | LR: 1.3e-03 [BEST]          
Epoch  10/100 | Train Loss: 1.0290 | Val Loss: 1.0119 | LR: 1.6e-03 [BEST]          
Epoch  11/100 | Train Loss: 0.9804 | Val Loss: 1.0125 | LR: 1.8e-03          
Epoch  12/100 | Train Loss: 0.9338 | Val Loss: 1.0113 | LR: 2.0e-03 [BES